[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_04_embeddings_lsa.ipynb)

# S12 · 04 — Embeddings: representar el significado con vectores (LSA)

🏠 Cuaderno para casa · ⏱ ≈ 50 min · Sprint 12 · UD 4.8

**Contexto TurisData.** TurisData quiere, dada una queja, encontrar rápidamente reseñas parecidas para ver si es un problema recurrente.

**Este cuaderno tiene 3 ejercicios y un mini-reto** y es **solo para casa**: no tiene versión «en clase». Es ampliación: hazlo en casa, a tu ritmo y por orden.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})
print("Listo ✔")

### ⚙️ A2 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
resenas = pd.read_csv(dato("resenas.csv"))

# Palabras vacías y de "relleno" (frases que abren muchas reseñas sin decir nada de la estancia). No quitamos «no».
VACIAS = {"a", "al", "con", "de", "del", "el", "en", "es", "esta", "estaba", "era", "fue", "la", "las", "lo", "los", "me", "mi",
          "muy", "nos", "para", "por", "que", "se", "su", "un", "una", "y", "o", "pasamos", "semana", "aquí", "escapada", "fin",
          "viajamos", "familia", "fuimos", "trabajo", "general", "todo", "estancia", "sin", "más"}

def limpiar(texto):
    palabras = re.sub(r"[^a-záéíóúüñ\s]", " ", texto.lower()).split()
    return " ".join(p for p in palabras if p not in VACIAS)

### 👀 A3 · EJEMPLO — TF-IDF: un vector largo y casi vacío

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué forma tiene la matriz TF-IDF?
2. ¿Qué proporción de la matriz son ceros?
3. ¿Por qué es un problema que casi todo sean ceros?

In [ ]:
tfidf = TfidfVectorizer(preprocessor=limpiar, min_df=2)
M = tfidf.fit_transform(resenas["texto"])
print("TF-IDF:", M.shape, "→ proporción de ceros:", round(1 - M.nnz / (M.shape[0] * M.shape[1]), 3))

### ✏️ A4 · EJERCICIO 1 de 3 — Reducir a 12 dimensiones con LSA

**Tu turno:** Reduce `M` a **12 dimensiones** con `TruncatedSVD(n_components=12, random_state=SEMILLA)`. Guarda el modelo en `lsa` y los vectores de las reseñas en `Z` (forma `(1500, 12)`). Guarda también en `var_explicada` la suma de la varianza explicada (`lsa.explained_variance_ratio_.sum()`).

**Debe salir:** `Forma de Z: (1500, 12) | varianza explicada con 12 ejes: 0.438` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué forma tiene `Z` y qué significa cada fila?
2. ¿Qué porcentaje de la variación recogen los 12 ejes?
3. ¿Qué se gana y qué se pierde al pasar de 105 a 12 columnas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: TruncatedSVD(n_components=12, random_state=SEMILLA); lsa.fit_transform(M)
lsa = ...
Z = ...
var_explicada = ...

print("Forma de Z:", Z.shape, "| varianza explicada con 12 ejes:", round(var_explicada, 3))
assert Z.shape == (len(resenas), 12), "Z debe tener 12 columnas: una por dimensión de LSA."
assert 0.2 < var_explicada < 1, "var_explicada debe ser un número entre 0 y 1 (suma del ratio de varianza)."
print("✔ Ejercicio 1 correcto")

### 👀 A5 · EJEMPLO — Las palabras con más peso en cada eje

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué palabras pesan más en el eje 4?
2. ¿Los ejes tienen nombre?
3. ¿Qué «tema» te sugiere el eje 2?

In [ ]:
terminos = np.array(tfidf.get_feature_names_out())
for i in range(5):
    peso = lsa.components_[i]
    arriba = terminos[np.argsort(peso)[-6:][::-1]]
    print(f"Eje {i + 1}: {', '.join(arriba)}")

### 👀 A6 · EJEMPLO — El coseno en un caso sencillo

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuánto vale el coseno entre vectores a 90°?
2. ¿Y entre `a` y `b`, que forman 45°?
3. ¿Por qué se usa el coseno y no la distancia?

In [ ]:
a, b, c = np.array([1, 0]), np.array([1, 1]), np.array([0, 1])
print("coseno(a, b) =", round(float(cosine_similarity([a], [b])[0, 0]), 3), "(45°)")
print("coseno(a, c) =", round(float(cosine_similarity([a], [c])[0, 0]), 3), "(90°, nada en común)")

### ✏️ A7 · EJERCICIO 2 de 3 — Las reseñas más parecidas a una dada

**Tu turno:** Completa `parecidas(i, k=3)`: devuelve los índices de las `k` reseñas más parecidas (por coseno sobre `Z`) a la reseña número `i`, **excluyendo la propia `i`**, de más a menos parecida.

**Debe salir:** la reseña 0 y 3 reseñas parecidas (las tres con «Vistas al mar increíbles») y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué tienen en común las tres reseñas parecidas a la reseña 0?
2. ¿Qué detalle importante de la reseña 0 no recogen?
3. ¿Para qué sirve `sim[i] = -1`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
def parecidas(i, k=3):
    # pista: cosine_similarity([Z[i]], Z)[0], poner -1 en la posición i y usar argsort descendente
    ...

vecinas = parecidas(0, 3)
print("Reseña 0:", resenas.loc[0, "texto"])
for j in vecinas:
    print("  ≈", resenas.loc[j, "texto"])
assert len(vecinas) == 3 and 0 not in vecinas, "parecidas debe devolver k índices distintos de i."
sim0 = cosine_similarity([Z[0]], Z)[0]
assert sim0[vecinas[0]] >= sim0[vecinas[1]] >= sim0[vecinas[2]], "Ordena de más a menos parecida."
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Dibujar las reseñas en 2 dimensiones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Conoce LSA el sentimiento de las reseñas?
2. ¿Por qué se solapan tanto los colores?
3. ¿Qué indica que aun así se agrupen algo?

In [ ]:
Z2 = TruncatedSVD(n_components=2, random_state=SEMILLA).fit_transform(M)
colores = {"positivo": "#3b8f5a", "neutro": "#999999", "negativo": "#c0463a"}
fig, ax = plt.subplots()
for s, col in colores.items():
    m = (resenas["sentimiento"] == s).values
    ax.scatter(Z2[m, 0], Z2[m, 1], s=12, alpha=.5, c=col, label=s)
ax.set_title("Reseñas en 2 dimensiones (LSA)"); ax.set_xlabel("Eje 1"); ax.set_ylabel("Eje 2"); ax.legend()
plt.show()

### 👀 A9 · EJEMPLO — Partir en entrenamiento y prueba y preparar TF-IDF

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Qué proporción de reseñas se reserva para probar?
2. ¿Para qué sirve `stratify`?
3. ¿Por qué se crea un vectorizador nuevo (`tfidf_e`)?

In [ ]:
X_ent, X_test, y_ent, y_test = train_test_split(resenas["texto"], resenas["sentimiento"], test_size=0.25,
                                                stratify=resenas["sentimiento"], random_state=SEMILLA)
tfidf_e = TfidfVectorizer(preprocessor=limpiar, min_df=2)

### ✏️ A10 · EJERCICIO 3 de 3 — LSA de 12 columnas frente a TF-IDF completo

**Tu turno:** Parte los datos en entrenamiento y prueba (ya hecho abajo). Ajusta `tfidf_e` y `lsa_e` **solo con `X_ent`**, entrena una regresión logística sobre los vectores LSA de entrenamiento y guarda en `acc_lsa` la exactitud en prueba.

**Debe salir:** `TF-IDF completo (105 columnas): 0.995`, `LSA (12 columnas)                : 0.891` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué exactitud logra LSA con 12 columnas y cuál el TF-IDF completo?
2. ¿Qué se paga por comprimir de 105 a 12 columnas?
3. ¿Qué error cometerías si ajustaras LSA con todas las reseñas antes de partir?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: fit_transform en entrenamiento y transform en prueba, tanto para tfidf_e como para lsa_e (12 componentes)
Me_ent = ...
Me_test = ...
lsa_e = ...
Ze_ent = ...
Ze_test = ...
clf_lsa = ...
acc_lsa = ...

clf_tfidf = LogisticRegression(max_iter=1000).fit(Me_ent, y_ent)
acc_tfidf = accuracy_score(y_test, clf_tfidf.predict(Me_test))
print(f"TF-IDF completo ({Me_ent.shape[1]} columnas): {acc_tfidf:.3f}")
print(f"LSA (12 columnas)                : {acc_lsa:.3f}")
assert Ze_ent.shape[0] == len(X_ent) and Ze_test.shape[0] == len(X_test), "Ajusta con X_ent y transforma X_test."
assert acc_lsa > 0.6, "acc_lsa debería superar 0.6; revisa que entrenes con Ze_ent/y_ent y evalúes con Ze_test."
print("✔ Ejercicio 3 correcto")

### ✏️ A11 · MINI-RETO — Buscador semántico de reseñas

**Tu turno:** Crea `buscar(consulta, k=3)`: convierte la consulta al espacio LSA (`tfidf.transform` y `lsa.transform`; usa `limpiar` como ya está en `tfidf`), calcula el coseno con `Z` y devuelve los `k` textos más parecidos. Prueba con `"no me dejó dormir el ruido"`.

**Debe salir:** 3 reseñas sobre ruido nocturno (por ejemplo, «… Ruido constante por la noche. No compensa el precio …») y ✔ Mini-reto: los resultados hablan de ruido

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Por qué se encuentran reseñas de ruido si la consulta no coincide con su texto?
2. ¿Por qué hay que pasar la consulta por `tfidf` y por `lsa`?
3. ¿Qué límite tiene este buscador frente a un modelo neuronal?

In [ ]:
def buscar(consulta, k=3):
    # pista: transforma la consulta con tfidf y luego con lsa; compara con Z y ordena de mayor a menor
    ...

hallazgos = buscar("no me dejó dormir el ruido")
for h in hallazgos:
    print("-", h)
assert len(hallazgos) == 3, "buscar debe devolver k textos."
assert any(("ruid" in h.lower() or "dormir" in h.lower() or "silencio" in h.lower() or "oye" in h.lower()) for h in hallazgos), \
    "Los resultados deberían hablar de ruido/dormir. Revisa que uses lsa.transform sobre la consulta y compares con Z."
print("✔ Mini-reto correcto")

### 👀 A12 · EJEMPLO — Embeddings neuronales (opcional, requiere internet)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Qué ventaja tiene un modelo preentrenado frente a LSA?
2. ¿Qué riesgo tiene?

In [ ]:
USAR_SENTENCE_TRANSFORMERS = False   # ← cámbialo a True solo si tienes internet (Colab)

if USAR_SENTENCE_TRANSFORMERS:
    # NO VERIFICADO: requiere internet
    %pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer
    modelo_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    E = modelo_st.encode(resenas["texto"].tolist(), show_progress_bar=False, normalize_embeddings=True)
    q = modelo_st.encode(["no me dejó dormir el ruido"], normalize_embeddings=True)
    mejores = np.argsort((E @ q.T).ravel())[::-1][:3]
    for i in mejores:
        print("-", resenas.loc[i, "texto"])
else:
    print("Parte opcional omitida (USAR_SENTENCE_TRANSFORMERS = False).")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `re`, `numpy`, `pandas`, `matplotlib` y, de `scikit-learn`, el vectorizador TF-IDF, `TruncatedSVD` (la reducción de dimensiones de LSA), la similitud del coseno, la regresión logística y la exactitud.
    - Fija la semilla (`SEMILLA`) y un estilo común para los gráficos.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

#### A2 · Preparación — Cargar las reseñas y la función de limpieza

**Contexto.** TurisData quiere, dada una queja, encontrar rápidamente reseñas parecidas para ver si es un problema recurrente.

- **Qué hace:**
    - `pd.read_csv(dato("resenas.csv"))` carga las 1 500 reseñas.
    - `VACIAS` es el conjunto de palabras vacías y de «relleno» (frases que abren muchas reseñas sin decir nada de la estancia); **no incluye «no»**.
    - `limpiar(texto)` pasa a minúsculas, quita signos y números y elimina las palabras vacías.
- **Qué debe salir:**
    - No imprime nada: deja preparados `resenas`, `VACIAS` y `limpiar`.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

## 1. Idea: de la bolsa de palabras al vector denso

#### A3 · Ejemplo — TF-IDF: un vector largo y casi vacío

TF-IDF da a cada reseña un vector **largo y casi vacío** (una columna por palabra, casi todo ceros: aquí 105 columnas y un 94 % de ceros; en un corpus real, miles de columnas).
Dos reseñas que hablan de lo mismo con palabras distintas ("ruidoso" / "no nos dejó dormir") quedan *lejos*.

Un **embedding** comprime eso a unos pocos números (aquí 12) donde las dimensiones capturan **temas** que van juntos.
**LSA** (*Latent Semantic Analysis*) lo hace con una descomposición matemática (SVD) que busca las combinaciones de
palabras que más varían; se la puede imaginar como "descubrir los ejes de conversación" del corpus.

- **Qué hace:**
    - `TfidfVectorizer(preprocessor=limpiar, min_df=2)` crea el vectorizador (usa tu `limpiar` y descarta las palabras que aparecen en menos de 2 reseñas).
    - `fit_transform(resenas["texto"])` da la matriz `M`: una fila por reseña y una columna por palabra.
    - La línea de `print` calcula qué proporción de `M` son ceros (`M.nnz` cuenta los valores distintos de cero).
- **Qué debe salir:**
    - `TF-IDF: (1500, 105) → proporción de ceros: 0.941`.
- **Cómo leerlo:**
    - Cada reseña es un vector de 105 números de los que el **94 %** son ceros: largo y casi vacío. En un corpus real serían miles de columnas.
- **Si falla:**
    - `NameError: name 'limpiar' is not defined` → falta ejecutar la celda anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene la matriz TF-IDF?* → **(1500, 105)**: 1 500 reseñas y 105 palabras.
2. *¿Qué proporción de la matriz son ceros?* → El **94,1 %**.
3. *¿Por qué es un problema que casi todo sean ceros?* → Porque el vector es **largo y poco eficiente**, y dos reseñas con palabras distintas aunque hablen de lo mismo quedan **lejos**.

</details>

#### A4 · Ejercicio 1 — Reducir a 12 dimensiones con LSA

- **Qué hace:**
    - `TruncatedSVD(n_components=12, random_state=SEMILLA)` crea el modelo de LSA que busca 12 «ejes» (`lsa`).
    - `lsa.fit_transform(M)` aprende los ejes y convierte cada reseña en un vector de 12 números (`Z`).
    - `float(lsa.explained_variance_ratio_.sum())` suma la parte de la variación que recogen los 12 ejes (`var_explicada`).
- **Qué debe salir:**
    - `Forma de Z: (1500, 12) | varianza explicada con 12 ejes: 0.438` y `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - Cada reseña pasa de 105 números a solo 12. Esos 12 ejes recogen el **43,8 %** de la variación del TF-IDF: se pierde información, y a cambio el vector es corto y denso.
- **Si falla:**
    - `Z debe tener 12 columnas: una por dimensión de LSA.` → `n_components` no es 12, o has guardado en `Z` el modelo en vez del resultado de `fit_transform`.
    - `var_explicada debe ser un número entre 0 y 1` → no has sumado `explained_variance_ratio_`.
    - `NameError: name 'M' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene `Z` y qué significa cada fila?* → **(1500, 12)**: cada fila es una reseña representada con **12 números**.
2. *¿Qué porcentaje de la variación recogen los 12 ejes?* → El **43,8 %**.
3. *¿Qué se gana y qué se pierde al pasar de 105 a 12 columnas?* → Se gana un vector **corto y denso**, más rápido y que agrupa palabras que van juntas; se pierde **información** (queda más de la mitad de la variación fuera).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
lsa = TruncatedSVD(n_components=12, random_state=SEMILLA)
Z = lsa.fit_transform(M)
var_explicada = float(lsa.explained_variance_ratio_.sum())
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. ¿Qué «significa» cada eje?

#### A5 · Ejemplo — Las palabras con más peso en cada eje

Cada eje de LSA es una combinación de palabras. Mirando las palabras con más peso en los primeros ejes
vemos qué "temas" ha encontrado **sin que le digamos nada**.

- **Qué hace:**
    - `terminos` es la lista de las 105 palabras del vocabulario.
    - Para cada uno de los 5 primeros ejes, `lsa.components_[i]` da el peso de cada palabra y `np.argsort(peso)[-6:][::-1]` elige las 6 con más peso.
- **Qué debe salir:**
    - Cinco líneas, por ejemplo `Eje 1: desayuno, limpio, precio, tranquilo, duerme, maravilla` y `Eje 3: total, noches, silencio, cinco, ubicación, perfecta`.
- **Cómo leerlo:**
    - Cada eje agrupa palabras que aparecen juntas: el eje 4 reúne `ubicación`, `playa`, `perfecta`, `minutos`; el 3, `silencio` y `ubicación`. Los ejes no tienen nombre propio: se **interpretan** mirando sus palabras (y no siempre son tan claros).
- **Si falla:**
    - `NameError: name 'lsa' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué palabras pesan más en el eje 4?* → **ubicación, playa, perfecta, cinco, minutos, precio**.
2. *¿Los ejes tienen nombre?* → **No**: hay que interpretarlos mirando las palabras que agrupan.
3. *¿Qué «tema» te sugiere el eje 2?* → **La habitación**: `habitación`, `limpieza`, `sencilla`, `correcta`, `impecable`.

</details>

## 3. Similitud del coseno

#### A6 · Ejemplo — El coseno en un caso sencillo

Para medir "cuánto se parecen" dos vectores se usa el **coseno del ángulo** entre ellos:
1 = misma dirección (muy parecidos), 0 = sin relación. No depende de la longitud del texto.

- **Qué hace:**
    - `a`, `b` y `c` son tres vectores de 2 números: `a` apunta a la derecha, `c` hacia arriba y `b` en diagonal.
    - `cosine_similarity([a], [b])` calcula el coseno del ángulo entre `a` y `b`; después, entre `a` y `c`.
- **Qué debe salir:**
    - `coseno(a, b) = 0.707 (45°)` y `coseno(a, c) = 0.0 (90°, nada en común)`.
- **Cómo leerlo:**
    - Cuanto más parecida es la dirección, más cerca de 1: 45° → 0,707 y 90° → 0. Los vectores de las reseñas se comparan igual, con 12 números en vez de 2.
- **Si falla:**
    - `NameError: name 'cosine_similarity' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el coseno entre vectores a 90°?* → **0**: no tienen nada en común.
2. *¿Y entre `a` y `b`, que forman 45°?* → **0,707**.
3. *¿Por qué se usa el coseno y no la distancia?* → Porque **no depende de la longitud** del texto: solo importa la dirección del vector.

</details>

#### A7 · Ejercicio 2 — Las reseñas más parecidas a una dada

- **Qué hace:**
    - `cosine_similarity([Z[i]], Z)[0]` calcula el coseno entre la reseña `i` y **todas** las demás (`sim`).
    - `sim[i] = -1` evita que la reseña salga como la más parecida a sí misma.
    - `np.argsort(sim)[::-1][:k]` ordena de mayor a menor parecido y se queda con los `k` primeros índices.
- **Qué debe salir:**
    - `Reseña 0: Escapada de fin de semana. La música del bar no nos dejó dormir. Vistas al mar increíbles. Estancia normal, sin más!`, tres reseñas «≈» (por ejemplo, `Fuimos por trabajo. Vistas al mar increíbles.`) y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Ojo con lo que ha captado el modelo: las tres vecinas coinciden en «Vistas al mar increíbles», pero **ninguna menciona la música del bar**. Con solo 12 ejes se conserva lo frecuente y se pierde el detalle raro.
- **Si falla:**
    - `parecidas debe devolver k índices distintos de i.` → no has puesto `sim[i] = -1`, y la propia reseña sale la primera.
    - `Ordena de más a menos parecida.` → falta el orden descendente (`[::-1]`).
    - `NameError: name 'Z' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tienen en común las tres reseñas parecidas a la reseña 0?* → Todas incluyen la frase **«Vistas al mar increíbles»**.
2. *¿Qué detalle importante de la reseña 0 no recogen?* → **La música del bar que no les dejó dormir**: es raro y los 12 ejes lo pierden.
3. *¿Para qué sirve `sim[i] = -1`?* → Para que la reseña **no se devuelva a sí misma** (su similitud consigo misma es 1).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def parecidas(i, k=3):
    sim = cosine_similarity([Z[i]], Z)[0]
    sim[i] = -1
    return list(np.argsort(sim)[::-1][:k])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Mapa de reseñas en 2 dimensiones

#### A8 · Ejemplo — Dibujar las reseñas en 2 dimensiones

Con solo 2 ejes podemos dibujar el corpus. Coloreamos por sentimiento (que LSA **no ha visto**).

- **Qué hace:**
    - `TruncatedSVD(n_components=2, ...)` reduce `M` a solo 2 ejes (`Z2`).
    - El bucle dibuja un punto por reseña, de color según su sentimiento (verde positivo, gris neutro, rojo negativo), aunque LSA **no ha visto** esa etiqueta.
- **Qué debe salir:**
    - Un gráfico de dispersión «Reseñas en 2 dimensiones (LSA)» con puntos verdes, grises y rojos.
- **Cómo leerlo:**
    - Los tres colores se solapan bastante: dos ejes recogen solo el 6 % de la variación. Aun así, en promedio las positivas quedan más a la derecha y abajo, y las negativas más arriba y a la izquierda: aparece algo de estructura **sin haberle dicho nada** de los sentimientos.
- **Si falla:**
    - `NameError: name 'M' is not defined` → falta el ejemplo de TF-IDF.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Conoce LSA el sentimiento de las reseñas?* → **No**: los colores se añaden después, solo para dibujar.
2. *¿Por qué se solapan tanto los colores?* → Porque dos ejes recogen **muy poca variación** (≈ 6 %): no bastan para separar bien los sentimientos.
3. *¿Qué indica que aun así se agrupen algo?* → Que los ejes captan **palabras que van juntas** y las reseñas de un mismo sentimiento comparten palabras.

</details>

## 5. Embeddings como entrada de un clasificador

#### A9 · Ejemplo — Partir en entrenamiento y prueba y preparar TF-IDF

Comparamos: regresión logística con **TF-IDF completo** frente a solo **12 números por reseña (LSA)**. Ojo: LSA es un modelo que se ajusta con datos; para no filtrar información, se ajusta **solo con el entrenamiento**.

- **Qué hace:**
    - `train_test_split(..., test_size=0.25, stratify=...)` separa un 25 % para probar, manteniendo la proporción de cada sentimiento.
    - `tfidf_e = TfidfVectorizer(preprocessor=limpiar, min_df=2)` crea un vectorizador **nuevo** que se ajustará solo con `X_ent`.
- **Qué debe salir:**
    - No imprime nada: deja preparados `X_ent`, `X_test`, `y_ent`, `y_test` y `tfidf_e`.
- **Si falla:**
    - `NameError: name 'limpiar' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué proporción de reseñas se reserva para probar?* → El **25 %** (`test_size=0.25`).
2. *¿Para qué sirve `stratify`?* → Para que entrenamiento y prueba tengan **la misma proporción** de cada sentimiento.
3. *¿Por qué se crea un vectorizador nuevo (`tfidf_e`)?* → Para ajustarlo **solo con el entrenamiento** y no filtrar información de la prueba.

</details>

#### A10 · Ejercicio 3 — LSA de 12 columnas frente a TF-IDF completo

- **Qué hace:**
    - `tfidf_e.fit_transform(X_ent)` aprende el vocabulario con el entrenamiento y lo convierte (`Me_ent`); `tfidf_e.transform(X_test)` convierte la prueba **sin reajustar** (`Me_test`).
    - `TruncatedSVD(n_components=12, ...)` (`lsa_e`) se ajusta igual: `fit_transform` con el entrenamiento (`Ze_ent`) y `transform` con la prueba (`Ze_test`).
    - `LogisticRegression(max_iter=1000).fit(Ze_ent, y_ent)` entrena el clasificador con los 12 números por reseña (`clf_lsa`) y `accuracy_score` da su exactitud en prueba (`acc_lsa`).
    - La comprobación entrena también el clasificador con el TF-IDF completo para compararlos.
- **Qué debe salir:**
    - `TF-IDF completo (105 columnas): 0.995`, `LSA (12 columnas)                : 0.891` y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Con 12 números se acierta un **89 %**; con las 105 columnas, un **99,5 %**. Comprimir ahorra tamaño pero cuesta exactitud. Si ajustaras LSA con **todas** las reseñas antes de partir, la prueba influiría en los ejes: sería **fuga de datos**.
- **Si falla:**
    - `Ajusta con X_ent y transforma X_test.` → has usado `fit_transform` también con la prueba, o has mezclado los conjuntos.
    - `acc_lsa debería superar 0.6` → has entrenado o evaluado con los vectores equivocados (usa `Ze_ent`/`y_ent` para entrenar y `Ze_test` para evaluar).
    - `NameError: name 'X_ent' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud logra LSA con 12 columnas y cuál el TF-IDF completo?* → **0,891** frente a **0,995**.
2. *¿Qué se paga por comprimir de 105 a 12 columnas?* → Unos **10 puntos de exactitud**.
3. *¿Qué error cometerías si ajustaras LSA con todas las reseñas antes de partir?* → **Fuga de datos**: la prueba influiría en los ejes y el resultado sería demasiado optimista.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
Me_ent = tfidf_e.fit_transform(X_ent)
Me_test = tfidf_e.transform(X_test)
lsa_e = TruncatedSVD(n_components=12, random_state=SEMILLA)
Ze_ent = lsa_e.fit_transform(Me_ent)
Ze_test = lsa_e.transform(Me_test)
clf_lsa = LogisticRegression(max_iter=1000).fit(Ze_ent, y_ent)
acc_lsa = accuracy_score(y_test, clf_lsa.predict(Ze_test))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Mini-reto: buscador semántico de reseñas

#### A11 · Mini-reto — Buscador semántico de reseñas

- **Qué hace:**
    - `tfidf.transform([consulta])` convierte la consulta en un vector TF-IDF (con el **mismo** vocabulario del corpus) y `lsa.transform(...)` la lleva al espacio de 12 ejes (`q`).
    - `cosine_similarity(q, Z)[0]` mide su parecido con todas las reseñas.
    - `np.argsort(sim)[::-1][:k]` selecciona las `k` más parecidas y se devuelven sus textos.
- **Qué debe salir:**
    - Tres reseñas que mencionan «Ruido constante por la noche» (la primera: `Pasamos una semana aquí. No compensa el precio. Ruido constante por la noche. Encontramos pelos en el baño...`) y la comprobación (sin mensaje de error).
- **Cómo leerlo:**
    - La consulta «no me dejó dormir el ruido» no aparece tal cual en ninguna reseña, y aun así se recuperan reseñas de ruido nocturno: el buscador funciona por **parecido de significado**, no por coincidencia exacta de texto.
- **Si falla:**
    - `buscar debe devolver k textos.` → no estás devolviendo los `k` primeros (o devuelves índices en vez de textos).
    - `Los resultados deberían hablar de ruido/dormir.` → has comparado la consulta sin pasarla por `lsa.transform`, o has ordenado de menor a mayor.
    - `NameError: name 'lsa' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué se encuentran reseñas de ruido si la consulta no coincide con su texto?* → Porque LSA compara **significado** (palabras que van juntas), no letras: «ruido» y «dormir» aparecen en reseñas parecidas.
2. *¿Por qué hay que pasar la consulta por `tfidf` y por `lsa`?* → Para que quede en el **mismo espacio** (mismo vocabulario y mismos 12 ejes) que las reseñas.
3. *¿Qué límite tiene este buscador frente a un modelo neuronal?* → LSA solo conoce **tu corpus**: si una consulta usa palabras que casi no aparecen en él, no la entenderá.

</details>

## 7. Opcional: embeddings neuronales

#### A12 · Ejemplo — Embeddings neuronales (opcional, requiere internet)

> **⚠️ NO VERIFICADO EN EL TALLER:** esta parte necesita descargar un modelo (≈ 470 MB según su página en Hugging Face, no comprobado en el taller) y instalar una librería.
> Ejecútala en **Google Colab** o en casa con conexión. Si no tienes internet, sáltala: el resto del notebook funciona sin ella.

La diferencia con LSA: un modelo neuronal ya *entrenado con millones de frases* entiende que "ruidoso" y "no pude dormir" se parecen
aunque en tu corpus casi no coincidan palabras. LSA solo conoce **tu** corpus.

- **Qué hace:**
    - `USAR_SENTENCE_TRANSFORMERS = False` desactiva la parte opcional; pásalo a `True` solo en Google Colab o con internet.
    - Con `True`, instala `sentence-transformers`, descarga un modelo multilingüe ya entrenado, calcula el embedding de cada reseña y de la consulta, y muestra las 3 más parecidas.
- **Qué debe salir:**
    - Con `False` (lo normal): `Parte opcional omitida (USAR_SENTENCE_TRANSFORMERS = False).`
- **Cómo leerlo:**
    - **No verificado en el taller**: necesita descargar un modelo de varios cientos de MB. Un modelo preentrenado entiende que «ruidoso» y «no pude dormir» se parecen aunque en tu corpus casi no coincidan palabras; LSA solo conoce **tu** corpus.
- **Si falla:**
    - Con `True` y sin internet: error al instalar la librería o al descargar el modelo → déjalo en `False`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué ventaja tiene un modelo preentrenado frente a LSA?* → Ha visto **millones de frases**, así que entiende que expresiones distintas pueden significar lo mismo.
2. *¿Qué riesgo tiene?* → **Depender de un tercero** y enviarle datos (privacidad), además de la descarga.

</details>

## Preguntas de reflexión
1. ¿Por qué los vectores LSA son "densos" y TF-IDF "dispersos"? ¿Qué ganamos y qué perdemos al comprimir de 105 columnas (miles en un corpus real) a 12 números?
2. Si hubiéramos ajustado el LSA con todas las reseñas antes de partir en train/test, ¿qué tipo de error cometeríamos?
3. Un embedding preentrenado (sentence-transformers) conoce el español general; ¿qué ventaja y qué riesgo tiene frente a LSA sobre nuestro corpus?
4. ¿Qué problemas de privacidad hay al enviar reseñas a un servicio externo para calcular embeddings?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Los vectores LSA son **densos** porque casi todos sus 12 números son distintos de cero; el TF-IDF es **disperso** porque cada reseña solo usa unas pocas de sus 105 palabras (94 % de ceros). Ganamos un vector corto, rápido y que agrupa palabras que van juntas; perdemos detalle (los 12 ejes recogen un 44 % de la variación) y algo de exactitud (89 % frente a 99,5 %).
2. Una **fuga de datos**: los ejes de LSA habrían «visto» las reseñas de prueba, así que la exactitud medida sería más optimista de lo que se obtendría con reseñas nuevas.
3. **Ventaja:** conoce el español general y entiende sinónimos y frases que en nuestro corpus casi no aparecen. **Riesgo:** depende de un tercero, no se ha ajustado a nuestro vocabulario (el turismo en Canarias) y no se puede revisar tan fácilmente cómo decide.
4. Las reseñas pueden contener **datos personales** (nombres, fechas, detalles de la estancia): enviarlas fuera exige base legal, **anonimizar antes**, enviar solo lo necesario y un contrato con el proveedor (RGPD).

</details>

## Resumen: qué has aprendido
- Un **embedding** representa un texto como un vector corto; "cerca" (coseno alto) = "parecido".
- **LSA** crea embeddings offline con TF-IDF + SVD; útil, rápido y explicable.
- Los embeddings sirven para **buscar por parecido** y como entrada de clasificadores.
- Los modelos neuronales preentrenados captan más significado, a costa de descargas y de depender de un tercero.